# Урок 01 - Вступ до агентів штучного інтелекту

Ласкаво просимо до першого уроку курсу **Агенти ШІ для початківців**!

**Агент ШІ** — це програма, яка використовує велику мовну модель (LLM) як механізм мислення і може виконувати *дії* у реальному світі — викликати API, запитувати бази даних або запускати код — щоб досягти цілі від імені користувача.

У цьому зошиті ви створите свого першого агента: **Туристичного агента**, який рекомендує місця для відпочинку. По дорозі ви навчитеся:

1. Підключатися до служби Azure AI Foundry Agent за допомогою **Microsoft Agent Framework**.
2. Надати агенту **інструмент** — просту функцію на Python, яку він може викликати.
3. Запустити агента та перевірити його відповідь.
4. Поступово отримувати відповідь агента токен за токеном.


## Налаштування

Перед запуском цієї записної книжки переконайтеся, що у вас є:

1. **Проєкт Azure AI Foundry** з розгорнутою моделлю чат-бота (наприклад, `gpt-4o-mini`).
2. **Вхід у систему через Azure CLI** — запустіть команду `az login` у вашому терміналі.
3. **Встановлені необхідні змінні оточення:**
   - `AZURE_AI_PROJECT_ENDPOINT` — кінцева точка вашого проєкту Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — назва вашої розгорнутої моделі.

Наведенна нижче клітинка встановлює необхідні пакунки Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Створення Вашого Першого Агента

Агенту потрібні дві речі:

- **Інструкції**, які кажуть йому *хто він* і *як поводитися* (системний запит).
- **Інструменти** — функції Python, прикрашені `@tool`, які агент може викликати для отримання інформації або виконання дій.

Нижче ми визначаємо простий інструмент, який повертає список популярних місць відпочинку. Агент буде використовувати цей інструмент, коли користувач запитає про рекомендації для подорожей.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потокові відповіді

Для більш інтерактивного досвіду ви можете **потоково** отримувати відповідь агента. Замість того, щоб чекати на повну відповідь, агент поступово видає частини тексту в міру їх генерації. Це особливо корисно в чат-інтерфейсах, де ви хочете відображати результат у реальному часі.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

У цьому уроці ви дізналися, як:

- **Створити провайдера**, який підключається до Azure AI Foundry Agent Service через `AzureAIProjectAgentProvider`.
- **Визначити інструмент** за допомогою декоратора `@tool`, щоб агент міг викликати ваші функції Python.
- **Запустити агента** з повідомленням користувача та вивести його відповідь.
- **Передавати відповіді потоково** для виводу в реальному часі.

У наступному уроці ми докладніше розглянемо агентські рамки та навчимося надавати агентам потужніші інструменти й можливості багатокрокового розмірковування.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:  
Цей документ було перекладено за допомогою автоматичного перекладача [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, просимо враховувати, що автоматизовані переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для важливої інформації рекомендується звертатися до професійного перекладу людиною. Ми не несемо відповідальності за будь-які непорозуміння чи неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
